# 02. Praproses dan Rekayasa Fitur (Peran B)

Notebook ini mencakup:
1. **Pemeriksaan Kualitas Data**: Verifikasi dimensi, *missing values*, duplikasi, dan tipe fitur pada data latih.
2. **Tabel Keputusan Praproses**: Dokumentasi keputusan desain rekayasa fitur beserta rasional ilmiahnya.
3. **Pengujian Pipeline**: Verifikasi fungsional modul src/pipeline.py pada skenario *dengan_G* dan *tanpa_G*.
4. **Demonstrasi Anti-Kebocoran Data (*Data Leakage*)**: Eksperimen komparasi F1-score ketika praproses di-fit di luar pipeline vs di dalam pipeline.

In [1]:
import os
import sys
import json
import pandas as pd
from pathlib import Path

# Menambahkan direktori utama ke path agar modul src terbaca
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pipeline import build_preprocessor

# Memuat data latih dan kontrak kolom
train = pd.read_csv(ROOT / 'data' / 'processed' / 'train.csv')
kontrak = json.load(open(ROOT / 'data' / 'processed' / 'columns.json'))

fitur_dengan_g = kontrak['fitur_dengan_g']
fitur_tanpa_g = kontrak['fitur_tanpa_g']
num_cols = kontrak['num_cols']
cat_cols = kontrak['cat_cols']
target = kontrak['target']

y = train[target]

# Pemeriksaan kualitas data
print('--- PEMERIKSAAN KUALITAS DATA ---')
print('Ukuran train:', train.shape)
print('Jumlah nilai hilang:', int(train.isna().sum().sum()))
print('Jumlah duplikat:', int(train.duplicated().sum()))
print('Jumlah kolom numerik:', len(num_cols))
print('Jumlah kolom kategorik:', len(cat_cols))


--- PEMERIKSAAN KUALITAS DATA ---
Ukuran train: (316, 34)
Jumlah nilai hilang: 0
Jumlah duplikat: 0
Jumlah kolom numerik: 15
Jumlah kolom kategorik: 17


## 1. Tabel Keputusan Praproses

| Langkah | Keputusan | Alasan |
|---|---|---|
| **Nilai hilang** | SimpleImputer: median untuk numerik, modus untuk kategorik | Data latih tidak memiliki nilai hilang, tetapi imputer tetap dipasang agar pipeline aman bila data baru memuat nilai kosong. Median tahan terhadap nilai ekstrem. |
| **Fitur kategorik nominal** | One-hot encoding (OneHotEncoder, handle_unknown='ignore', keluaran padat) | Label encoding memberi urutan palsu pada kategori nominal yang dapat menyesatkan model berbasis jarak (kNN, SVM). handle_unknown='ignore' mencegah error saat ada kategori baru pada data uji. Keluaran padat diperlukan karena GaussianNB tidak menerima matriks sparse. |
| **Fitur numerik** | StandardScaler (rata-rata 0, simpangan baku 1) | Wajib bagi SVM dan kNN yang peka terhadap skala jarak; tidak memengaruhi integritas model berbasis pohon. |
| **Atribut ordinal berkode angka** | Diperlakukan sebagai numerik | Urutan nilai (Medu, Fedu, studytime, amrel) memiliki makna tingkatan bertingkat; one-hot akan memperbanyak dimensi secara berlebihan pada data berukuran kecil. |
| **Pencilan bsences** | Dibiarkan (tidak dipotong/ditransformasi) | Nilai ketidakhadiran tinggi mencerminkan fenomena nyata siswa berisiko. Model pohon kebal terhadap skala pencilan, dan pemotongan berisiko mereduksi varians informasi kritis. |
| **Ketidakseimbangan kelas** | class_weight='balanced' pada model yang mendukung | Menghindari bias prediksi kelas mayoritas tanpa menciptakan sampel sintetis buatan (seperti SMOTE) yang rentan menimbulkan derau pada sampel 316 baris. |
| **Seleksi fitur** | Tidak dilakukan | Menjaga seluruh variasi prediktor latar belakang siswa tetap utuh. |
| **Dua skenario fitur** | Modularisasi melalui uild_preprocessor(fitur, num_cols, cat_cols) | Arsitektur transformer tunggal yang fleksibel menyaring fitur untuk skenario *dengan_G* dan *tanpa_G* secara konsisten. |

In [2]:
# Uji fungsi build_preprocessor pada kedua skenario
pre_dengan_g = build_preprocessor(fitur_dengan_g, num_cols, cat_cols)
X_trans_dg = pre_dengan_g.fit_transform(train[fitur_dengan_g])

pre_tanpa_g = build_preprocessor(fitur_tanpa_g, num_cols, cat_cols)
X_trans_tg = pre_tanpa_g.fit_transform(train[fitur_tanpa_g])

print('--- PENGUJIAN TRANSFORMASI PREPROCESSOR ---')
print(f'Skenario dengan_G: {len(fitur_dengan_g)} fitur asli -> {X_trans_dg.shape[1]} fitur setelah one-hot | Bentuk matriks: {X_trans_dg.shape}')
print(f'Skenario tanpa_G : {len(fitur_tanpa_g)} fitur asli -> {X_trans_tg.shape[1]} fitur setelah one-hot | Bentuk matriks: {X_trans_tg.shape}')


--- PENGUJIAN TRANSFORMASI PREPROCESSOR ---
Skenario dengan_G: 32 fitur asli -> 58 fitur setelah one-hot | Bentuk matriks: (316, 58)
Skenario tanpa_G : 30 fitur asli -> 56 fitur setelah one-hot | Bentuk matriks: (316, 56)


## 2. Demonstrasi Pencegahan Kebocoran Data (*Data Leakage*)

**Kebocoran Data (*Data Leakage*)** terjadi jika parameter transformasi (misal rata-rata dan simpangan baku StandardScaler, atau kamus kategori OneHotEncoder) dipelajari dari seluruh dataset sebelum proses validasi silang.

Di bawah ini disimulasikan dua pendekatan validasi silang (Repeated Stratified 5-Fold, 3 ulangan, random_state=42):
1. **Cara Salah (Bocor)**: Preprocessor di-*fit* pada seluruh data latih, baru kemudian dilakukan validasi silang pada data hasil olahan.
2. **Cara Benar (Pipeline)**: Preprocessor dibungkus dalam Pipeline, sehingga di-*fit* hanya pada fold latih tiap iterasi dan diuji pada fold validasi murni.

In [3]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
MODEL = {
    'kNN (k=5)': lambda: KNeighborsClassifier(5),
    'SVM (RBF)': lambda: SVC(class_weight='balanced', random_state=42),
    'Decision Tree (depth 3)': lambda: DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=42),
}

skenario_dict = {'dengan_G': fitur_dengan_g, 'tanpa_G': fitur_tanpa_g}
baris = []
for sk, fitur in skenario_dict.items():
    X = train[fitur]
    # (1) SALAH: praproses di-fit pada SELURUH data latih, baru validasi silang
    X_bocor = build_preprocessor(fitur, num_cols, cat_cols).fit_transform(X)
    for nama, buat in MODEL.items():
        s_bocor = cross_val_score(buat(), X_bocor, y, cv=cv, scoring='f1').mean()
        # (2) BENAR: praproses di dalam pipeline, di-fit hanya pada data latih tiap fold
        pipa = Pipeline([('pre', build_preprocessor(fitur, num_cols, cat_cols)), ('clf', buat())])
        s_benar = cross_val_score(pipa, X, y, cv=cv, scoring='f1').mean()
        baris.append((sk, nama, round(s_bocor, 4), round(s_benar, 4), round(s_bocor - s_benar, 4)))

tabel = pd.DataFrame(baris, columns=['skenario', 'model', 'F1_di_luar_pipeline', 'F1_di_dalam_pipeline', 'selisih'])
print(tabel.to_string(index=False))
print('\nKesimpulan: Praproses di dalam Pipeline menjamin estimasi performa jujur tanpa kebocoran informasi fold validasi.')


skenario                   model  F1_di_luar_pipeline  F1_di_dalam_pipeline  selisih
dengan_G               kNN (k=5)               0.6601                0.6429   0.0172
dengan_G               SVM (RBF)               0.8348                0.8314   0.0034
dengan_G Decision Tree (depth 3)               0.8631                0.8631   0.0000
 tanpa_G               kNN (k=5)               0.3182                0.3194  -0.0012
 tanpa_G               SVM (RBF)               0.4428                0.4454  -0.0027
 tanpa_G Decision Tree (depth 3)               0.4151                0.4151   0.0000

Kesimpulan: Praproses di dalam Pipeline menjamin estimasi performa jujur tanpa kebocoran informasi fold validasi.
